# Detached EEG Pipeline (Joint Standalone)

This notebook is specifically designed to run the **Joint Training (RQ2)** experiment on Google Colab, utilizing the `joint_trials.npz` dataset.

> **⚡ Auto-Reconnect Support**: This notebook has two-level checkpointing. If Colab disconnects, simply re-run all cells from the top — the pipeline automatically resumes from the last completed fold *without* repeating the expensive GPU pre-transformation step.

### 🔄 Restart / Recovery Instructions

Google Colab Pay-as-you-go runtimes can disconnect at any time. This notebook is designed to recover automatically.

#### On first run
Run all cells top-to-bottom as normal.

#### After a disconnection
1. **Re-run all cells from the top** (Runtime → Run all).
2. The pipeline will automatically:
   - Load the **feature matrix cache** from `results/cache_joint/` on Drive.
   - Resume the **LOSO loop** from the last completed fold (reads `results/loso_results_joint.json`).

#### Cache locations on Google Drive
| File | Contents |
|------|----------|
| `results/cache_joint/feature_matrix_{m}.npy` | Pre-computed feature matrices (10 files) |
| `results/cache_joint/X_all.npy` etc. | Formatted EEG data |
| `results/cache_joint/cache_meta.json` | Cache config hash for invalidation |
| `results/loso_results_joint.json` | Fold results (updated after each fold) |


In [7]:
# Mount Google Drive if you have uploaded your dataset here
from google.colab import drive
drive.mount('/content/drive')

# Change to the directory where your detachment-eeg code and data live
import os
os.chdir('/content/drive/MyDrive/Colab')
print("Working directory:", os.getcwd())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Working directory: /content/drive/MyDrive/Colab


In [8]:
import os
import sys

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

# Install Dependencies
!pip install mne numpy scikit-learn pyts torch matplotlib sktime==0.30.0
!pip install git+https://github.com/gon-uri/detach_rocket --quiet

  Preparing metadata (setup.py) ... done


In [9]:
# ── Colab Keep-Alive ────────────────────────────────────────────────────────
from IPython.display import display, Javascript

display(Javascript('''
function ClickConnect(){
    console.log("[keep-alive] Simulating activity to prevent idle timeout.");
    var btn = document.querySelector("colab-toolbar-button#connect");
    if (btn) btn.click();
}
setInterval(ClickConnect, 60000);
'''))
print("✓ Keep-alive activated (fires every 60s). Re-run this cell after each reconnect.")

<IPython.core.display.Javascript object>

✓ Keep-alive activated (fires every 60s). Re-run this cell after each reconnect.


### Imports and Setup

In [10]:
import mne
import numpy as np
import json
import os
from typing import Dict, Any, List
from sklearn.metrics import accuracy_score, classification_report

# Add the project root directory to Python path for robust imports
import sys
project_root = '/content/drive/MyDrive/Colab'
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Now import from src.utl
from src.utl.config import load_config

# Import Detach Rocket
try:
    from detach_rocket.detach_classes import DetachEnsemble
except ImportError:
    try:
        from detach_rocket.detach_rocket import DetachEnsemble
    except ImportError:
        print("Warning: detach_rocket not installed correctly. Using placeholder.")
        DetachEnsemble = None

mne.set_log_level("WARNING")

### Pipeline Class Definition

In [ ]:
class TaskEEGPipeline:
    """
    EEG Classification Pipeline utilizing Detach-Rocket Ensemble for the Joint Dataset.
    """
    def __init__(self, config_path: str = "config.yml"):
        self.config = load_config(config_path)

        # Load experiment settings
        # For joint standalone, we assume the joint path directly. If not in config, fallback to default.
        self.data_path = self.config.get('data', {}).get('joint', './data/processed/joint_dataset/joint_trials.npz')

        self.split_seed = self.config.get('experiment', {}).get('seed', 42)
        self.n_splits = self.config.get('experiment', {}).get('n_splits', 5)
        self.binary_mode = self.config.get('experiment', {}).get('binary_classification', True)

        self.all_subjects = []
        self.splits: List[Dict[str, Any]] | None = None
        self.processed_folds_data: List[Dict[str, Any]] = []
        self.completed_subjects: List[str] = []

        self._check_cuda()

    def _check_cuda(self):
        try:
            import torch
            cuda_available = torch.cuda.is_available()
            print(f"\n" + "="*30)
            print(f"CUDA STATUS CHECK")
            print(f"="*30)
            print(f"CUDA Available: {'YES' if cuda_available else 'NO'}")
            if cuda_available:
                print(f"Device Name: {torch.cuda.get_device_name(0)}")
                print(f"VRAM Total: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
            print(f"="*30 + "\n")
        except Exception as e:
            print(f"\n[!] Error checking CUDA: {e}\n")

    def initialize(self):
        """Prepare splits from the .npz subject list."""
        print(f"Initializing joint pipeline with data at: {self.data_path}")
        if not os.path.exists(self.data_path):
            print(f"[!] Joint dataset not found at {self.data_path}")
            return

        data = np.load(self.data_path)
        self.all_subjects = data['subject_list'].tolist()
        print(f"Discovered {len(self.all_subjects)} unique subjects in the joint dataset.")

        # ── Level 2 Checkpoint: Load completed fold results ────────────────────
        results_path = "results/loso_results_joint.json"
        if os.path.exists(results_path) and os.path.getsize(results_path) > 0:
            try:
                with open(results_path, "r") as f:
                    all_runs_results = json.load(f)
                    if isinstance(all_runs_results, list) and all_runs_results:
                        last_run_results = all_runs_results[-1]  # Most recent run
                        self.processed_folds_data = last_run_results.get("folds", [])
                        self.completed_subjects = [fold["subject"] for fold in self.processed_folds_data]
                        print(f"Found {len(self.completed_subjects)} completed folds from previous run.")
            except Exception as e:
                print(f"Error reading '{results_path}': {e}. Starting fresh.")

        remaining_loso_subjects = [sub for sub in self.all_subjects if sub not in self.completed_subjects]

        if not remaining_loso_subjects:
            print("All subjects already processed. No new folds to run.")
            self.splits = []
            return

        print(f"Remaining subjects to process: {len(remaining_loso_subjects)}.")

        self.splits = []
        for i, test_sub in enumerate(remaining_loso_subjects):
            train_subs = [s for s in self.all_subjects if s != test_sub]
            self.splits.append({
                "train_subjects": train_subs,
                "test_subjects": [test_sub],
                "val_idx": self.all_subjects.index(test_sub)
            })

        print(f"Generated {len(self.splits)} new LOSO cross-validation folds to run.")

    def _get_cache_key(self):
        import hashlib
        model_params = self.config.get('model', {}).get('params', {})
        key_parts = {
            "num_models": model_params.get('num_models', 10),
            "num_kernels": model_params.get('num_kernels', 10000),
            "data_path": self.data_path,
        }
        key_str = json.dumps(key_parts, sort_keys=True)
        return hashlib.md5(key_str.encode()).hexdigest()

    def _save_precomputed_cache(self, feature_matrices):
        import datetime
        cache_dir = "results/cache_joint"
        os.makedirs(cache_dir, exist_ok=True)

        sep = "=" * 55
        print(f"\n{sep}")
        print("SAVING FEATURE MATRIX CACHE TO DRIVE")
        print(sep)

        for m, F in enumerate(feature_matrices):
            np.save(os.path.join(cache_dir, f"feature_matrix_{m}.npy"), F)
            print(f"  Saved feature matrix {m+1}/{len(feature_matrices)} — shape: {F.shape}.")

        cache_meta = {
            "cache_key": self._get_cache_key(),
            "num_models": len(feature_matrices),
            "created_at": datetime.datetime.utcnow().isoformat()
        }
        with open(os.path.join(cache_dir, "cache_meta.json"), "w") as f:
            json.dump(cache_meta, f, indent=4)
        print(f"{sep}\n")

    def _load_precomputed_cache(self):
        cache_dir = "results/cache_joint"
        meta_path = os.path.join(cache_dir, "cache_meta.json")

        if not os.path.exists(meta_path):
            print("No feature matrix cache found. Will compute from scratch.")
            return None

        try:
            with open(meta_path, "r") as f:
                cache_meta = json.load(f)
        except Exception as e:
            print(f"Could not read cache metadata ({e}). Recomputing from scratch.")
            return None

        current_key = self._get_cache_key()
        if cache_meta.get("cache_key") != current_key:
            print("[!] Config changed since cache was built — cache invalidated. Recomputing.")
            return None

        model_params = self.config.get('model', {}).get('params', {})
        num_models = model_params.get('num_models', 10)

        print(f"\nCACHE HIT — Loading pre-computed feature matrices")
        try:
            feature_matrices = []
            for m in range(num_models):
                F = np.load(os.path.join(cache_dir, f"feature_matrix_{m}.npy"))
                feature_matrices.append(F)
                print(f"  Loaded feature matrix {m+1}/{num_models} — shape: {F.shape}")
            print("All feature matrices loaded. Pre-transformation step SKIPPED.\n")
            return feature_matrices
        except Exception as e:
            print(f"Error loading cache files ({e}). Recomputing from scratch.")
            return None

    def run(self):
        self.initialize()

        if not self.all_subjects:
            print("Pipeline not initialized or no data found. Aborting.")
            return

        if not self.splits and not self.processed_folds_data:
            print("No splits to process and no previous data to report. Aborting.")
            return

        from detach_rocket.detach_classes import PytorchMiniRocketMultivariate, DetachMatrix
        import torch
        device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

        # Always load the base data arrays directly from the .npz for joint training
        print(f"Loading raw data from {self.data_path}...")
        data = np.load(self.data_path)
        X_all = data['X']
        y_all = data['y']
        s_indices = data['subject_indices']
        print(f"Loaded {X_all.shape[0]} trials across {len(self.all_subjects)} subjects.")

        # ── Level 1 Checkpoint: Feature Matrix Cache ───────────────────────────
        feature_matrices = self._load_precomputed_cache()

        if feature_matrices is None:
            model_params = self.config.get('model', {}).get('params', {})
            num_models = model_params.get('num_models', 10)
            num_kernels = model_params.get('num_kernels', 10000)

            print(f"\nInitializing {num_models} Detach-Rocket transformers (Target: {num_kernels} kernels each)...")
            transformers = []
            feature_matrices = []

            for m in range(num_models):
                print(f"Creating Feature Matrix for Model {m+1}/{num_models}...")
                transformer = PytorchMiniRocketMultivariate(num_features=num_kernels, device=device)
                transformer.fit(X_all)
                F = transformer.transform(X_all).numpy()
                transformers.append(transformer)
                feature_matrices.append(F)

            self._save_precomputed_cache(feature_matrices)

        # ── Level 2 Checkpoint: Per-Fold Results ───────────────────────────────
        print("\nStarting Optimized LOSO Cross-Validation Loop...")

        subject_predictions = [fold['y_pred'] for fold in self.processed_folds_data]
        subject_true_labels = [fold['y_true'] for fold in self.processed_folds_data]
        results = list(self.processed_folds_data)

        for i, split in enumerate(self.splits):
            test_sub_id = split['test_subjects'][0]
            print(f"\n--- Fold {len(results)+1}/{len(self.all_subjects)} (Testing: {test_sub_id}) ---")

            test_sub_idx_in_all = split['val_idx']
            train_mask = s_indices != test_sub_idx_in_all
            test_mask = s_indices == test_sub_idx_in_all

            y_train = y_all[train_mask]
            y_test = y_all[test_mask]

            model_outputs = []
            model_weights = []

            for m in range(len(feature_matrices)):
                F_train = feature_matrices[m][train_mask]
                F_test = feature_matrices[m][test_mask]

                model = DetachMatrix(
                    trade_off=self.config.get('experiment', {}).get('trade_off', 0.1)
                )
                model.fit(F_train, y_train)
                y_pred_m = model.predict(F_test)
                model_outputs.append(y_pred_m)
                model_weights.append(model._acc_train)

            model_outputs = np.array(model_outputs).T
            weights = np.array(model_weights)

            y_pred_probas = (model_outputs * weights).sum(axis=1) / weights.sum()
            y_pred_trials = (y_pred_probas >= 0.5).astype(int)

            y_pred_subject = 1 if np.mean(y_pred_trials) >= 0.5 else 0
            y_true_subject = y_test[0]

            subject_predictions.append(y_pred_subject)
            subject_true_labels.append(y_true_subject)

            results.append({
                "subject": test_sub_id,
                "y_true": int(y_true_subject),
                "y_pred": int(y_pred_subject),
                "trial_accuracy": float(accuracy_score(y_test, y_pred_trials))
            })

            print(f"Subject Prediction: {y_pred_subject} (True: {y_true_subject})")

            self._save_results_checkpoint(
                subject_predictions, subject_true_labels, results
            )

        if subject_predictions:
            final_acc = accuracy_score(subject_true_labels, subject_predictions)
            print(f"\nFinal Subject-Level Accuracy (LOSO): {final_acc*100:.2f}%")
            print("\nClassification Report:")
            print(classification_report(subject_true_labels, subject_predictions, target_names=['Control', 'AD']))
            print("\nFinal results saved to results/loso_results_joint.json")

    def _save_results_checkpoint(self, subject_predictions, subject_true_labels, current_folds_results):
        os.makedirs("results", exist_ok=True)
        json_path = "results/loso_results_joint.json"

        current_run_results = {
            "config": self.config,
            "model_params": {
                "num_models": self.config.get('model', {}).get('params', {}).get('num_models', 10),
                "num_kernels": self.config.get('model', {}).get('params', {}).get('num_kernels', 10000),
                "trade_off": self.config.get('experiment', {}).get('trade_off', 0.1)
            },
            "subject_level_accuracy": accuracy_score(subject_true_labels, subject_predictions) if len(subject_predictions) > 0 else 0.0,
            "paper_target_accuracy": 0.8615,
            "folds": current_folds_results
        }

        all_runs_results = []
        if os.path.exists(json_path) and os.path.getsize(json_path) > 0:
            try:
                with open(json_path, "r") as f:
                    existing_data = json.load(f)
                    if isinstance(existing_data, list):
                        all_runs_results = existing_data
                    else:
                        all_runs_results.append(existing_data)
            except Exception as e:
                print(f"Error reading '{json_path}': {e}. Starting fresh.")

        if len(all_runs_results) > 0:
            last_saved_run_config = all_runs_results[-1].get("config")
            if last_saved_run_config == self.config:
                all_runs_results[-1] = current_run_results
                experiment_number = all_runs_results[-1].get("experiment_number", len(all_runs_results))
            else:
                experiment_number = len(all_runs_results) + 1
                current_run_results["experiment_number"] = experiment_number
                all_runs_results.append(current_run_results)
        else:
            experiment_number = 1
            current_run_results["experiment_number"] = experiment_number
            all_runs_results.append(current_run_results)

        with open(json_path, "w") as f:
            json.dump(all_runs_results, f, indent=4)
        print(f"Checkpoint saved for experiment {experiment_number} to {json_path} "
              f"(processed {len(current_folds_results)}/{len(self.all_subjects)} folds).")


: 

### Run the Pipeline

In [12]:
if __name__ == "__main__":
    pipeline = TaskEEGPipeline()
    pipeline.run()


CUDA STATUS CHECK
CUDA Available: YES
Device Name: Tesla T4
VRAM Total: 14.56 GB

Initializing joint pipeline with data at: ./data/processed/joint_dataset/joint_trials.npz
Discovered 65 unique subjects in the joint dataset.
Found 10 completed folds from previous run.
Remaining subjects to process: 55.
Generated 55 new LOSO cross-validation folds to run.
Loading raw data from ./data/processed/joint_dataset/joint_trials.npz...
Loaded 14649 trials across 65 subjects.

CACHE HIT — Loading pre-computed feature matrices
  Loaded feature matrix 1/10 — shape: (14649, 9996)
  Loaded feature matrix 2/10 — shape: (14649, 9996)
  Loaded feature matrix 3/10 — shape: (14649, 9996)
  Loaded feature matrix 4/10 — shape: (14649, 9996)
  Loaded feature matrix 5/10 — shape: (14649, 9996)
  Loaded feature matrix 6/10 — shape: (14649, 9996)
  Loaded feature matrix 7/10 — shape: (14649, 9996)
  Loaded feature matrix 8/10 — shape: (14649, 9996)
  Loaded feature matrix 9/10 — shape: (14649, 9996)
  Loaded fe

: 

: 

In [ ]:
import json
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
import pandas as pd
import os

def load_results(json_path="results/loso_results_joint.json"):
    if not os.path.exists(json_path):
        print(f"Error: Could not find {json_path}. Please run the pipeline first.")
        return None
    with open(json_path, 'r') as f:
        data = json.load(f)
        if isinstance(data, list):
            if data:
                return data[-1]  # Return the most recent run
            else:
                print(f"Error: '{json_path}' is an empty list. No results to display.")
                return None
        elif isinstance(data, dict):
            return data  # Already a single run
        else:
            print(f"Error: '{json_path}' contains unexpected data format. Expected a dict or a list of dicts.")
            return None

def plot_confusion_matrix(results, ax=None):
    y_true = [fold['y_true'] for fold in results['folds']]
    y_pred = [fold['y_pred'] for fold in results['folds']]

    cm = confusion_matrix(y_true, y_pred)
    df_cm = pd.DataFrame(cm, index=['Control', 'AD'], columns=['Control', 'AD'])

    if ax is None:
        fig, ax = plt.subplots(figsize=(6, 5))

    sns.heatmap(df_cm, annot=True, fmt='g', cmap='Blues', ax=ax, annot_kws={"size": 16})
    ax.set_title('Subject-Level Confusion Matrix (Joint Dataset)')
    ax.set_ylabel('True Label')
    ax.set_xlabel('Predicted Label')

    return ax.figure

def plot_trial_accuracy(results, ax=None):
    df = pd.DataFrame(results['folds'])

    # Sort by true label so Controls and ADs are grouped
    df.sort_values(by=['y_true', 'trial_accuracy'], ascending=[True, False], inplace=True)

    if ax is None:
        fig, ax = plt.subplots(figsize=(10, 5))

    colors = ['skyblue' if y == 0 else 'salmon' for y in df['y_true']]
    bars = ax.bar(df['subject'], df['trial_accuracy'] * 100, color=colors)

    ax.axhline(50, color='red', linestyle='--', alpha=0.5, label='Random Chance')
    ax.set_title('Trial-Level Accuracy per Subject (Joint Dataset)')
    ax.set_ylabel('Accuracy (%)')
    ax.set_xticks(range(len(df['subject'])))
    ax.set_xticklabels(df['subject'], rotation=45, ha='right')

    # Custom legend
    from matplotlib.patches import Patch
    legend_elements = [Patch(facecolor='skyblue', label='Control'),
                       Patch(facecolor='salmon', label='AD')]
    ax.legend(handles=legend_elements)

    plt.tight_layout()
    return ax.figure

def main():
    results = load_results()
    if not results:
        return

    os.makedirs('results/figures', exist_ok=True)

    print(f"Loaded results! Final Subject-Level Accuracy: {results['subject_level_accuracy']*100:.2f}%")
    print(f"Target Accuracy (Original Paper): {results.get('paper_target_accuracy', 0.8615)*100:.2f}%\n")

    # 1. Plot Confusion Matrix
    print("Generating Confusion Matrix...")
    fig_cm = plot_confusion_matrix(results)
    plt.show(fig_cm)

    # 2. Plot Trial Accuracy
    print("Generating Trial Accuracy Chart...")
    fig_acc = plot_trial_accuracy(results)
    plt.show(fig_acc)

    print("\nVisualization complete!")

if __name__ == "__main__":
    main()